# Climate Trend Analyzer - Walkthrough Notebook

End-to-end student portfolio notebook. It mirrors `main.py` step by step:

1. **Virtual dataset** - physically plausible daily climate records, 1990-2024
2. **Cleaning + features** - duplicates, impossible values, missing cells, baseline normals
3. **EDA** - summary tables, climatology, correlations
4. **Trend analysis** - OLS + Theil-Sen slopes, Mann-Kendall test
5. **Anomaly detection** - z-score + percentile + IsolationForest fusion
6. **Forecasting** - Holt-Winters with an honest 24-month backtest
7. **Interactive 3D views** - Plotly surfaces and animated charts

> Run order: top to bottom. First execution takes ~1 minute.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Image, Markdown
except ModuleNotFoundError:  # running outside jupyter - keep cells executable
    def display(x):
        print(x)
    def Image(filename):
        return filename
    def Markdown(x):
        return x

from src import config as C
from src.simulation import generate_climate_dataset, save_dataset
from src.preprocessing import run_preprocessing
from src import eda, trends, anomaly, forecast, visuals

pd.set_option('display.max_columns', 50)
print('project root :', C.PROJECT_ROOT)
print('symbols      :', ' '.join(C.SYMBOLS.values()))

## 1 - Virtual dataset generation
Real climate servers (IMD / NOAA) are not publicly scriptable for students,
so we simulate records with a warming trend, monsoon seasonality, rising CO2,
autocorrelated weather noise, injected extreme events and sensor errors.

In [ ]:
raw, events = generate_climate_dataset()
save_dataset(raw)
display(raw.head())
print('rows        :', len(raw))
print('regions     :', raw['region'].nunique())
print('span        :', raw['date'].min().date(), '->', raw['date'].max().date())
print('events logged:', len(events))
events.groupby('event_type').size()

## 2 - Cleaning + feature engineering
Duplicates removed, humidity > 100 % and negative rainfall repaired,
missing cells interpolated per region, then calendar features,
1991-2020 baseline normals, anomalies and rolling means are added.

In [ ]:
df, quality = run_preprocessing()
display(pd.Series(quality).to_frame('value'))
display(df.head(3))
print('final rows :', len(df), '| missing cells:', quality['missing_cells_after'])

## 3 - Exploratory data analysis

In [ ]:
res = eda.run_eda(df)
display(eda.summary_tables(df)['yearly_summary'].head(10))
display(eda.summary_tables(df)['correlation_matrix'])
Image(filename=str(res['figures']['yearly_temperature']))

In [ ]:
Image(filename=str(res['figures']['monthly_climatology']))

## 4 - Trend analysis
OLS slope per decade, robust Theil-Sen slope and the non-parametric
Mann-Kendall test on annual aggregates.

In [ ]:
tres = trends.run_trend_analysis(df)
display(tres['trends'][tres['trends'].variable == 'temperature_c'])
display(tres['shifts'])
Image(filename=str(tres['figures']['anomaly_trend']))

## 5 - Anomaly / extreme-event detection
Three methods are fused: |z| of the deseasonalised temperature anomaly,
99th-percentile rainfall per region-month, and IsolationForest.
A day survives if two methods agree or if it sits inside a persistent run.

In [ ]:
ares = anomaly.run_anomaly_detection(df)
display(ares['metrics'].T)
display(ares['summary'])
display(ares['top'].head(10))
Image(filename=str(ares['figures']['anomaly_timeline']))

## 6 - Forecasting (Holt-Winters + 24-month backtest)

In [ ]:
fres = forecast.run_forecast(df, method='holt_winters')
display(fres['accuracy'])
display(fres['projections'].head(8))
Image(filename=str(fres['figures']['forecast']))

## 7 - Interactive 3D views (Plotly inside the notebook)

In [ ]:
flagged = ares['flagged']
surf = visuals.fig_surface_temperature(flagged, region=C.REGIONS[0][0])
surf.show()
pillar = visuals.fig_climate_pillar(df)
pillar.show()

In [ ]:
scatter = visuals.fig_scatter3d_anomalies(flagged, sample=4000)
scatter.show()
anim = visuals.fig_animated_yearly(df)
anim.show()

## 8 - Automated report
The same report the pipeline writes to `outputs/reports/climate_report.md`.

In [ ]:
from src.report import build_report, save_report
content = build_report(
    clean_df=df, quality=quality,
    trends=tres['trends'],
    anomaly_metrics=ares['metrics'], anomaly_summary=ares['summary'],
    forecast_accuracy=fres['accuracy'], projections=fres['projections'])
path = save_report(content)
print('report ->', path)
display(Markdown(content[:2500]))